# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [4]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** One line-item on one order because lines is the repeated list. Everything else (order, vendor_id) gets copied down.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [5]:
import pandas as pd

orders = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order','vendor_id'],
    sep='.'
)

assert len(orders) == 4
assert orders['qty'].sum() == 7

orders


,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [8]:
vendors = pd.json_normalize(vendors_json)

joined = orders.merge(
    vendors,
    on='vendor_id',
    how='left',
    indicator=True,
    validate='many_to_one'
)

unmatched_vendor = joined.loc[joined['_merge'] == 'left_only', 'vendor_id'].unique()

unmatched_units = joined.loc[joined['_merge'] == 'left_only', 'qty'].sum()

print("Unmatched vendor:", unmatched_vendor)
print("Units attached:", unmatched_units)


Unmatched vendor: ['V-07']
Units attached: 3


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [10]:
by_vendor_name = (
    joined.groupby('name', dropna=False)['qty']
          .sum()
          .sort_values(ascending=False)
)

print(by_vendor_name)

# Comment: why the unmatched vendor appears as NaN


name
Hoos Burgers    3
NaN             3
Cav Merch       1
Name: qty, dtype: int64


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [11]:
zone_totals = (
    joined.groupby('zone', dropna=False)['qty']
          .sum()
          .sort_values(ascending=False)
)

print(zone_totals)


zone
A      4
NaN    3
Name: qty, dtype: int64


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [12]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

import pandas as pd

# Only keep records that actually have a 'lines' list
safe_records = [rec for rec in ragged_json if 'lines' in rec]

ragged = pd.json_normalize(
    safe_records,
    record_path='lines',
    meta=['order', 'vendor_id'],
    sep='.'
)

ragged


,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Fries,NaN,6,V-01


### Q6 — Validate

In [13]:
# assert len(lines) == 4
# assert lines['qty'].sum() == 7
# assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')


checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per line item is useful because it turns every item sold into its own measurbale unit. That makes questions like "*Which item sold the most?*, "*How many total units did vendor x self?*", or "*What's the revenue by product?*" trivial. If you kept one row per order, you'd have the lists inside cells, and every aggregation would require custom unpacking logic. Line-item shape makes grouping, summing, and joining effortless.

b) A missing quantity mean "we genuinely don't know the number," while a quantity of zero means "the number is known and it is zero." If you treat missing as zero, you silently convert bad or incomplete data into valid data. Downstream, totals become wrong, validation checks pass when they shouldn't and you lose the ability to detect malformerd payloads. Keeping missing as NaN preserves the signal that something is wrong.